# Hydroclimate tables (PC) — Collection 01

MapBiomas Chile — Chile Water.

Prepares CSV inputs for `complete_hydroclim_gee.js` (GEE).

**Inputs** (put under `input/`):
- `BIG_GRIDS_CHILE.shp` (cartas)
- `caudales_cartas.xlsx` (sheet `Caudal_mensual`)

| Step | What it does |
|------|----------------|
| 0 | Grid catalog from shapefile |
| 1 | `forcing_type` / `forcing_source` in `grid_attributes_hydro` |
| 2 | Flow climatology + annual anomaly tables |
| 3 | Export to `export_for_gee/` |

**GEE:** upload CSVs as Table assets, then run `complete_hydroclim_gee.js`.
Macrozone is assigned in GEE from `MZ_CHILE` (not on PC).
Flow grids: `forcing_type == flow`; others use CR2MET precip in GEE.

In [ ]:
from pathlib import Path
import geopandas as gpd
import numpy as np
import pandas as pd

# Notebook directory (HYDROCLIMATE/)
DIR = Path('.').resolve()
# Inputs: place shapefile + Excel under input/, or edit paths.
SHP_PATH = DIR / 'input' / 'BIG_GRIDS_CHILE.shp'
EXCEL_CAUDAL = DIR / 'input' / 'caudales_cartas.xlsx'
OUT_DIR = DIR / 'export_for_gee'
OUT_DIR.mkdir(parents=True, exist_ok=True)

YEAR_START, YEAR_END = 1998, 2025
Z_W, Z_D, Z_P = 0.5, -0.5, 2.0
AGGREGATE_STATIONS_MEAN = True

## Paso 0 — Cartas desde shapefile


In [ ]:
# Macrozona: NO se asigna en PC. Ver complete_hydroclim_gee.js (vector MZ_CHILE).

def classify_position(name, lon):
    s = name.split("-")[-1]
    if s in ("V","X") and lon >= -71.5: return "coast"
    if s in ("Y","Z") and lon <= -70.0: return "andes"
    return "inland"

gdf = gpd.read_file(SHP_PATH).to_crs(4326)
print(len(gdf), "cartas")

rows = []
for _, row in gdf.iterrows():
    name = str(row["grid_name"]).strip()
    c = row.geometry.centroid
    lo, la = float(c.x), float(c.y)
    rows.append(dict(
        grid_name=name,
        forcing_type="precip",
        forcing_source="CR2MET",
        position=classify_position(name, lo),
        bolivian_winter=0,
        lon_centroid=round(lo,4), lat_centroid=round(la,4),
        station_codes="", notes="",
    ))
grid_attrs = pd.DataFrame(rows).sort_values(
    ["lat_centroid","lon_centroid"], ascending=[False, True]
)
grid_attrs.head()


## Paso 1 — Forzante desde Excel (Pilar)


In [ ]:
caudal_raw = pd.read_excel(EXCEL_CAUDAL, sheet_name="Caudal_mensual")
flow_set = set(caudal_raw["grid_name"].dropna().unique())

grid_attrs["forcing_type"] = grid_attrs.grid_name.apply(
    lambda g: "flow" if g in flow_set else "precip"
)
grid_attrs["forcing_source"] = grid_attrs.forcing_type.map(
    {"flow": "DGA", "precip": "CR2MET"}
)

codes = caudal_raw.groupby("grid_name")["CODIGO_ESTACION"].apply(
    lambda s: ";".join(sorted({str(x) for x in s.dropna()}))
)
grid_attrs["station_codes"] = grid_attrs.apply(
    lambda r: codes.get(r.grid_name, "") if r.grid_name in flow_set else "",
    axis=1,
)
grid_attrs.loc[grid_attrs.forcing_type == "flow", "notes"] = (
    "Caudal DGA; lista Pilar; caudales_cartas.xlsx"
)

print("precip:", (grid_attrs.forcing_type == "precip").sum())
print("flow:  ", (grid_attrs.forcing_type == "flow").sum())
grid_attrs[grid_attrs.forcing_type == "flow"][
    ["grid_name", "forcing_type", "forcing_source", "station_codes"]
]


## Paso 2 — Climatología y anomalía (caudal)


In [ ]:
def p33_p67(v):
    s = sorted(v)
    return s[3], s[8]

def bw(m):
    return 1 if m >= 10 or m <= 3 else 0

def ystate(z):
    if z >= Z_W: return "W"
    if z <= Z_D: return "D"
    return "N"

df = caudal_raw.copy()
df["Fecha"] = pd.to_datetime(df["Fecha"])
df = df[(df.Fecha.dt.year >= YEAR_START) & (df.Fecha.dt.year <= YEAR_END)]
df["year"], df["month"] = df.Fecha.dt.year, df.Fecha.dt.month

if AGGREGATE_STATIONS_MEAN:
    monthly = df.groupby(["grid_name", "year", "month"], as_index=False)["Caudal"].mean()
else:
    st = df.groupby("grid_name")["CODIGO_ESTACION"].first()
    df = df.merge(st.rename("st"), left_on="grid_name", right_index=True)
    monthly = df[df.CODIGO_ESTACION == df.st].drop(columns="st")

clim_rows = []
for gname, gdf in monthly.groupby("grid_name"):
    clim = gdf.groupby("month")["Caudal"].mean()
    months = [float(clim.get(m, 0)) for m in range(1, 13)]
    p33, p67 = p33_p67(months)
    max_m = int(np.argmax(months)) + 1
    row = dict(
        grid_name=gname, forcing_type="flow", forcing_source="DGA",
        p33=p33, p67=p67, max_month=max_m, bolivian_winter_detected=bw(max_m),
    )
    for i, v in enumerate(months, 1):
        row[f"m{i}"] = v
    clim_rows.append(row)
seasonal_climatology_flow = pd.DataFrame(clim_rows)

annual = monthly.groupby(["grid_name", "year"], as_index=False)["Caudal"].sum()
annual = annual.rename(columns={"Caudal": "annual_value"})
anom_rows = []
for gname, gdf in annual.groupby("grid_name"):
    mu, sd = gdf.annual_value.mean(), gdf.annual_value.std(ddof=0)
    for _, r in gdf.iterrows():
        z = (r.annual_value - mu) / sd if sd > 0 else 0
        anom_rows.append(dict(
            grid_name=gname, year=int(r.year),
            annual_value=r.annual_value, annual_pp=r.annual_value,
            z_grid=z, year_state=ystate(z),
            problem_flag=1 if abs(z) >= Z_P else 0,
            forcing_type="flow", forcing_source="DGA",
        ))
annual_anomaly_flow = pd.DataFrame(anom_rows)
seasonal_climatology_flow.head()


## Control de cobertura (caudal DGA)

Avisa si alguna carta `flow` no tiene los 28 años (1998–2025) en el Excel.

In [ ]:
FULL_YEARS = set(range(YEAR_START, YEAR_END + 1))
flow_names = sorted(grid_attrs.loc[grid_attrs.forcing_type == "flow", "grid_name"])
years_in_monthly = monthly.groupby("grid_name")["year"].apply(set)

gaps = []
for g in flow_names:
    have = years_in_monthly.get(g, set())
    missing = sorted(FULL_YEARS - have)
    if missing:
        gaps.append({
            "grid_name": g,
            "n_missing": len(missing),
            "missing_years": ",".join(map(str, missing)),
        })

def _export_complete(exclude):
    complete = grid_attrs[
        grid_attrs.grid_name.isin(flow_names) & ~grid_attrs.grid_name.isin(exclude)
    ][["grid_name"]].assign(notes="28 años caudal 1998-2025")
    complete.to_csv(OUT_DIR / "cartas_flow_completas.csv", index=False)
    print("Piloto flow completo:", len(complete), "cartas -> cartas_flow_completas.csv")
    return complete

if gaps:
    gap_df = pd.DataFrame(gaps)
    print("ATENCION: cartas flow con años sin dato en caudales_cartas.xlsx")
    display(gap_df)
    gap_df.to_csv(OUT_DIR / "datos_faltantes_caudal.csv", index=False)
    _export_complete(gap_df.grid_name)
    n_max = len(flow_names) * len(FULL_YEARS)
    print("Filas anomalia:", len(annual_anomaly_flow), "de maximo", n_max)
else:
    print("Cobertura OK: 28 años en todas las cartas flow.")
    _export_complete([])



## Paso 3 — Exportar


In [ ]:
grid_attrs.to_csv(OUT_DIR / "grid_attributes_hydro.csv", index=False)
seasonal_climatology_flow.to_csv(OUT_DIR / "seasonal_climatology_flow.csv", index=False)
annual_anomaly_flow.to_csv(OUT_DIR / "annual_anomaly_flow.csv", index=False)

# Serie mensual 1998-2025 para curvas agua vs caudal en GEE (gee_revisar_gap_precip.js)
monthly_flow_merged = monthly.rename(columns={"Caudal": "flow_value"}).copy()
monthly_flow_merged["forcing_type"] = "flow"
monthly_flow_merged["forcing_source"] = "DGA"
monthly_flow_merged = monthly_flow_merged[
    ["grid_name", "year", "month", "flow_value", "forcing_type", "forcing_source"]
]
monthly_flow_merged.to_csv(OUT_DIR / "monthly_flow_merged.csv", index=False)

REF = DIR.parent / "grid_attributes_hydro.csv"
grid_attrs.to_csv(REF, index=False)

print("Listo:", OUT_DIR.resolve())
print("Referencia:", REF)
print("Filas monthly_flow_merged:", len(monthly_flow_merged))
